In [1]:
import numpy as np
import math

from Util.Problems import Problem, solution
import Util.math_functions as mathf

class P023(Problem):
    number = 23
    title = "Non-Abundant Sums"
    description = """<p>A perfect number is a number for which the sum of its proper divisors is exactly equal to the number. For example, the sum of the proper divisors of $28$ would be $1 + 2 + 4 + 7 + 14 = 28$, which means that $28$ is a perfect number.</p><p>A number $n$ is called deficient if the sum of its proper divisors is less than $n$ and it is called abundant if this sum exceeds $n$.</p><p>As $12$ is the smallest abundant number, $1 + 2 + 3 + 4 + 6 = 16$, the smallest number that can be written as the sum of two abundant numbers is $24$. By mathematical analysis, it can be shown that all integers greater than $28123$ can be written as the sum of two abundant numbers. However, this upper limit cannot be reduced any further by analysis even though it is known that the greatest number that cannot be expressed as the sum of two abundant numbers is less than this limit.</p><p>Find the sum of all the positive integers which cannot be written as the sum of two abundant numbers.</p>"""
    upper_limit = 28123

In [2]:
p = P023()
p.describe()

## Problem 23: Non-Abundant Sums

<p>A perfect number is a number for which the sum of its proper divisors is exactly equal to the number. For example, the sum of the proper divisors of $28$ would be $1 + 2 + 4 + 7 + 14 = 28$, which means that $28$ is a perfect number.</p><p>A number $n$ is called deficient if the sum of its proper divisors is less than $n$ and it is called abundant if this sum exceeds $n$.</p><p>As $12$ is the smallest abundant number, $1 + 2 + 3 + 4 + 6 = 16$, the smallest number that can be written as the sum of two abundant numbers is $24$. By mathematical analysis, it can be shown that all integers greater than $28123$ can be written as the sum of two abundant numbers. However, this upper limit cannot be reduced any further by analysis even though it is known that the greatest number that cannot be expressed as the sum of two abundant numbers is less than this limit.</p><p>Find the sum of all the positive integers which cannot be written as the sum of two abundant numbers.</p>

### Solution notes
We start by creating a list of all abundant numbers up to the upper limit. We then loop throug all numbers, for each number n loop through the abundant numbers up to n/2. Subtract the abundant number from n and check if the remainder is also in the list of abundant numbers. If so, continue to the next n, if not, continue looping through the abundant numbers, if none of them produce a result, add this to a running total of numbers which cannot be expressed as the sum of 2 abundant numbers.

In [3]:
@solution(P023, first=True, max_tests= 0, make_fast=True, warmup_args=(P023.upper_limit, ))
def brute_force_list_check(upper_limit):
    abundant_numbers = []
    for i in range(12, upper_limit):
        if (sum(mathf.get_divisors(i)) - i) > i:
            abundant_numbers.append(i)
    total_sum = 0
    for i in range(1, upper_limit):
        for abundant_number in abundant_numbers:
            if abundant_number > (i / 2):
                total_sum += i
                break
            remainder = i - abundant_number
            if remainder in abundant_numbers:
                break
    return total_sum

In [4]:
p.test_once("brute_force_list_check")

4179871 found after a separate test in 8023.820200 ms by brute_force_list_check (first)


Now, instead of checking if the remainder is in the list of abundant numbers, we check manually if it is an abundant number. Because lists are so very slow, this speeds up the process a lot.

In [5]:
@solution(P023, max_tests= 0, make_fast=True, warmup_args=(P023.upper_limit, ))
def brute_force_manual_check(upper_limit):
    abundant_numbers = []
    for i in range(12, upper_limit):
        if (sum(mathf.get_divisors(i)) - i) > i:
            abundant_numbers.append(i)
    total_sum = 0
    for i in range(1, upper_limit):
        for abundant_number in abundant_numbers:
            if abundant_number > (i / 2):
                total_sum += i
                break
            remainder = i - abundant_number
            if (sum(mathf.get_divisors(remainder)) - remainder) > remainder:
                break
    return total_sum

In [6]:
p.test_once("brute_force_manual_check")

4179871 found after a separate test in 1102.047100 ms by brute_force_manual_check


Next, we track the abundant numbers not with a list, but a boolean array. For each index, this array says whether it is abundant or not. Using the nonzero() function, we can get the indices of all abundant numbers to cycle through when checking any n. When checking if the remainder is abundant, however, this is now a single boolean lookup in an array.

In [7]:
@solution(P023, max_tests= 100, make_fast=True, warmup_args=(P023.upper_limit, ))
def bool_array(upper_limit):
    check_abundance = np.zeros(upper_limit, dtype=np.bool_)
    for i in range(12, upper_limit):
        if (sum(mathf.get_divisors(i)) - i) > i:
            check_abundance[i] = True

    abundant_numbers = check_abundance.nonzero()[0]
    total_sum = 0
    for i in range(1, upper_limit):
        for abundant_number in abundant_numbers:
            if abundant_number > (i / 2):
                total_sum += i
                break
            remainder = i - abundant_number
            if check_abundance[remainder]:
                break
    return total_sum

In [8]:
p.test_once("bool_array")

4179871 found after a separate test in 11.571900 ms by bool_array


The biggest bottleneck was now initially checking for abundance in all numbers. To optimise this, we no longer use the get_divisors() function. Instead, we create an altered version of it, which only needs to track the sum of all divisors. As soon as this sum becomes bigger than the number itself, we can already mark this as an abundant number and move on.

In [9]:
@solution(P023, max_tests= 100, make_fast=True, warmup_args=(P023.upper_limit, ))
def faster_creation_bool_array(upper_limit):
    check_abundance = np.zeros(upper_limit, dtype=np.bool_)
    for i in range(12, upper_limit):
        number = 2
        divisor_sum = 1
        while number * number <= i:
            if i % number == 0:
                divisor_sum += number
                if divisor_sum > i:
                    check_abundance[i] = True
                    break
                number_pair = i // number
                if number_pair != number:
                    divisor_sum += number_pair
                    if divisor_sum > i:
                        check_abundance[i] = True
                        break
            number += 1

    abundant_numbers = check_abundance.nonzero()[0]
    total_sum = 0
    for i in range(1, upper_limit):
        for abundant_number in abundant_numbers:
            if abundant_number > (i / 2):
                total_sum += i
                break
            remainder = i - abundant_number
            if check_abundance[remainder]:
                break
    return total_sum

In [10]:
p.test_all()

4179871 found after 100 tests in 10.708551 ms by bool_array
4179871 found after a separate test in 8023.820200 ms by brute_force_list_check (first)
4179871 found after a separate test in 1102.047100 ms by brute_force_manual_check
4179871 found after 100 tests in 6.379489 ms by faster_creation_bool_array
